# xG Shot Map Generator
Load any Opta match JSON → get a full shot DataFrame with all model outputs → build shot maps.

**Models**: Pre-trained on 31,738 WSL shots (2015–2026), XGBoost + isotonic calibration  
**Outputs per shot:**

| Column | Meaning |
|---|---|
| `xg` | Pre-shot goal probability |
| `psxg` | Post-shot xG (where the ball was aimed) |
| `placement_score` | Corner placement quality (0–100) |
| `shot_power` | Heuristic shot power index (0–100) |
| `placement_quality` | psxG − xG (shot placement added value) |
| `finishing_luck` | is_goal − psxG |
| `is_outfield_block` | Q82-confirmed outfield defender block |
| `is_keeper_save` | No Q82 → goalkeeper save |

**Instructions:**
1. Set `JSON_PATH` to any Opta match JSON
2. Set `MODEL_DIR` to the `xg_output/` folder (contains trained `.pkl` files)
3. Optionally set `BADGE_DIR` and `OUT_DIR`
4. Run all cells

In [ ]:
# ── CONFIGURE ─────────────────────────────────────────────────────────────────
JSON_PATH  = '/content/drive/MyDrive/Project-Beth-Mead/WSL 2024-2025/DONE/some_match.json'
MODEL_DIR  = '/content/drive/MyDrive/Project-Beth-Mead/xg_output'   # folder with model_xg.pkl etc.
BADGE_DIR  = None    # folder with <TeamName>.png badges, or None
OUT_DIR    = None    # folder to save PNGs/CSV, or None to display only
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
import os, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import matplotlib.image as mpimg
from mplsoccer import VerticalPitch
import joblib

%matplotlib inline
plt.rcParams['figure.dpi'] = 130

# ── Geometry constants (Opta) ─────────────────────────────────────────────────
GOAL_X = 100.0; GOAL_Y_LEFT = 44.0; GOAL_Y_RIGHT = 56.0
GOAL_Y_CENTRE = 50.0; GOAL_WIDTH = 12.0; GOAL_H_HIGH = 62.0

Q = dict(
    PENALTY=9, HEADER=15, RIGHT_FOOT=20, LEFT_FOOT=72,
    REGULAR_PLAY=22, FAST_BREAK=23, SET_PIECE=24, FROM_CORNER=25,
    FREE_KICK=26, DIRECT_FK=28, VOLLEY=108, DEFLECTION=133,
    PULL_BACK=195, BIG_CHANCE=233, FIRST_TIME=200,
    GOAL_Y=102, GOAL_HEIGHT=231, SAVE_END_X=146, SAVE_END_Y=147,
    UNDER_PRESSURE=18, INTENTIONAL=154, BODY_SIDE=56,
    BLOCKED=82,
)

# ── Helpers ───────────────────────────────────────────────────────────────────
def open_angle(x, y):
    shot  = np.array([x, y], dtype=float)
    v1 = np.array([GOAL_X, GOAL_Y_LEFT])  - shot
    v2 = np.array([GOAL_X, GOAL_Y_RIGHT]) - shot
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0: return 0.0
    return float(np.degrees(np.arccos(np.clip(np.dot(v1,v2)/(n1*n2),-1,1))))

def dist_to_goal(x, y):
    return float(np.sqrt((GOAL_X-x)**2 + (y-GOAL_Y_CENTRE)**2))

def placement_score(gy_norm, gh_norm):
    if pd.isna(gy_norm) or pd.isna(gh_norm): return np.nan
    gy = float(np.clip(gy_norm,-1,1)); gh = float(np.clip(gh_norm,0,1))
    lat = abs(gy); vert = abs(gh-0.40) / max(0.40, 0.60)
    return float(np.clip(np.sqrt(0.6*lat**2+0.4*vert**2)*100, 0, 100))

def goal_zone(goal_y, goal_h):
    if pd.isna(goal_y) or pd.isna(goal_h): return 'Unknown'
    h = 'Top' if goal_h >= GOAL_H_HIGH else 'Bottom'
    s = 'Left' if goal_y < 47.5 else ('Right' if goal_y > 52.5 else 'Centre')
    return f'{h} {s}'

def get_q(event, qid):
    for q in event.get('qualifier', []):
        if q['qualifierId'] == qid: return q.get('value', 1)
    return None

def has_q(event, qid):
    return any(q['qualifierId'] == qid for q in event.get('qualifier', []))

def safe_float(v):
    try: return float(v)
    except: return np.nan

print('Helpers loaded ✓')

## 1 · Load match & run models

In [ ]:
# ── Load Opta match JSON ──────────────────────────────────────────────────────
def load_opta_json(path):
    with open(path, 'r', encoding='utf-8-sig') as f:
        data = json.load(f)
    events = data.get('liveData', {}).get('event', data.get('event', []))
    rows = []
    for e in events:
        tid = str(e.get('typeId'))
        if tid not in ('13','14','15','16'): continue
        x = safe_float(e.get('x')); y = safe_float(e.get('y'))
        if np.isnan(x) or np.isnan(y): continue
        gy_raw  = safe_float(get_q(e, Q['GOAL_Y']))
        gh_raw  = safe_float(get_q(e, Q['GOAL_HEIGHT']))
        gy_norm = (gy_raw - GOAL_Y_CENTRE)/(GOAL_WIDTH/2) if not np.isnan(gy_raw) else np.nan
        gh_norm = gh_raw/100.0 if not np.isnan(gh_raw) else np.nan
        if not np.isnan(gy_norm) and not np.isnan(gh_norm):
            gf_dist = np.sqrt(gy_norm**2+(gh_norm-0.35)**2)
            corner  = int(abs(gy_norm)>0.55 or gh_norm>0.60)
            ps      = placement_score(gy_norm, gh_norm)
        elif not np.isnan(gy_norm):
            gf_dist = abs(gy_norm); corner = int(abs(gy_norm)>0.55); ps = abs(gy_norm)*100
        else:
            gf_dist = corner = ps = np.nan
        body_side = str(get_q(e, Q['BODY_SIDE']) or '').strip()
        is_rf = has_q(e, Q['RIGHT_FOOT']); is_lf = has_q(e, Q['LEFT_FOOT'])
        weak_foot = int((is_rf and body_side=='Left') or (is_lf and body_side=='Right'))
        dist = dist_to_goal(x, y); angle = open_angle(x, y); y_sym = abs(y - GOAL_Y_CENTRE)
        rows.append({
            'match_file': os.path.basename(path),
            'player_id': str(e.get('playerId','')), 'player_name': e.get('playerName',''),
            'contestant_id': str(e.get('contestantId','')),
            'type_id': int(tid), 'period_id': int(e.get('periodId') or 0),
            'time_min': int(e.get('timeMin') or 0),
            'x': x, 'y': y, 'y_sym': y_sym,
            'distance': dist, 'log_distance': np.log(max(dist,0.5)),
            'angle': angle, 'open_angle': angle, 'angle_sin': np.sin(np.radians(angle)),
            'in_six_yard': int(x>=94.2 and 36.8<=y<=63.2),
            'in_penalty_box': int(x>=83.0 and 21.1<=y<=78.9),
            'central_y': int(y_sym < GOAL_WIDTH/2),
            'dist_to_post': abs(y_sym - GOAL_WIDTH/2),
            'goal_y_raw': gy_raw, 'goal_y_norm': gy_norm,
            'goal_h_raw': gh_raw, 'goal_h_norm': gh_norm,
            'goal_frame_dist': gf_dist, 'corner_zone': corner,
            'placement_score': ps, 'goal_zone': goal_zone(gy_raw, gh_raw),
            'is_header': int(has_q(e,Q['HEADER'])),
            'is_right_foot': int(is_rf), 'is_left_foot': int(is_lf), 'weak_foot': weak_foot,
            'is_volley': int(has_q(e,Q['VOLLEY'])), 'is_deflected': int(has_q(e,Q['DEFLECTION'])),
            'is_first_time': int(has_q(e,Q['FIRST_TIME'])), 'is_big_chance': int(has_q(e,Q['BIG_CHANCE'])),
            'is_fast_break': int(has_q(e,Q['FAST_BREAK'])), 'is_from_corner': int(has_q(e,Q['FROM_CORNER'])),
            'is_free_kick': int(has_q(e,Q['DIRECT_FK'])), 'is_penalty': int(has_q(e,Q['PENALTY'])),
            'is_set_piece': int(has_q(e,Q['SET_PIECE'])), 'is_open_play': int(has_q(e,Q['REGULAR_PLAY'])),
            'is_pull_back': int(has_q(e,Q['PULL_BACK'])), 'under_pressure': int(has_q(e,Q['UNDER_PRESSURE'])),
            'is_intentional': int(has_q(e,Q['INTENTIONAL'])),
            'is_goal': int(tid=='16'), 'is_on_target': int(tid in ('15','16')),
            'is_post': int(tid=='14'), 'is_blocked': int(tid=='15'),
            'is_outfield_block': int(tid=='15' and has_q(e, Q['BLOCKED'])),
            'is_keeper_save':    int(tid=='15' and not has_q(e, Q['BLOCKED'])),
        })
    return pd.DataFrame(rows)

# ── Load pre-trained models ───────────────────────────────────────────────────
xg_model    = joblib.load(os.path.join(MODEL_DIR, 'model_xg.pkl'))
psxg_model  = joblib.load(os.path.join(MODEL_DIR, 'model_psxg.pkl'))
meta        = joblib.load(os.path.join(MODEL_DIR, 'model_meta.pkl'))
psxg_meta   = joblib.load(os.path.join(MODEL_DIR, 'model_psxg_meta.pkl'))
XG_FEATURES   = meta['features']
PSXG_FEATURES = psxg_meta['features']
PEN_XG        = float(meta['pen_xg'])
print(f'Models loaded  (xG features: {len(XG_FEATURES)}  psxG features: {len(PSXG_FEATURES)})')

# ── Load match & predict ──────────────────────────────────────────────────────
shots = load_opta_json(JSON_PATH)

pen_mask = shots['is_penalty'] == 1
nonpen   = shots[~pen_mask].copy()

X_base = nonpen[XG_FEATURES].fillna(0).astype(float)
nonpen['xg'] = xg_model.predict_proba(X_base)[:, 1]

X_ps = nonpen[PSXG_FEATURES].fillna(0).astype(float)
nonpen['psxg'] = psxg_model.predict_proba(X_ps)[:, 1]
# For shots without placement data, fall back to xg
has_pl = nonpen[['goal_y_norm','goal_h_norm']].notna().any(axis=1)
nonpen.loc[~has_pl, 'psxg'] = nonpen.loc[~has_pl, 'xg']

shots.loc[~pen_mask, 'xg']   = nonpen['xg'].values
shots.loc[pen_mask,  'xg']   = PEN_XG
shots.loc[~pen_mask, 'psxg'] = nonpen['psxg'].values
shots.loc[pen_mask,  'psxg'] = PEN_XG

# Derived metrics
shots['placement_quality'] = shots['psxg'] - shots['xg']
shots['finishing_luck']    = shots['is_goal'] - shots['psxg']

def shot_power(row):
    p = 50.0
    if row.get('is_volley', 0):      p += 25
    if row.get('is_first_time', 0):  p += 12
    if row.get('is_header', 0):      p -= 8
    if row.get('weak_foot', 0):      p -= 12
    if row.get('under_pressure', 0): p -= 8
    if row.get('is_deflected', 0):   p += 5
    return float(np.clip(p, 0, 100))

shots['shot_power'] = shots.apply(shot_power, axis=1)

def outcome_label(row):
    if row.get('is_goal', 0):    return 'Goal'
    if row.get('is_blocked', 0): return 'On Target'
    if row.get('is_post', 0):    return 'Post'
    return 'Missed'

shots['outcome'] = shots.apply(outcome_label, axis=1)

print(f'Shots: {len(shots)}  |  Goals: {shots.is_goal.sum()}  |  xG: {shots.xg.sum():.2f}  |  psxG: {shots.psxg.sum():.2f}')
print(f'Outfield blocks: {shots.is_outfield_block.sum()}  |  Keeper saves: {shots.is_keeper_save.sum()}')

## 2 · Team name lookup

In [ ]:
# Extract match title and team names from filename and JSON
filename_base = os.path.splitext(os.path.basename(JSON_PATH))[0]
try:
    date_str, match_str = filename_base.split('_', 1)
    home_name, away_name = match_str.split(' - ', 1)
except ValueError:
    date_str = ''; home_name = 'Home'; away_name = 'Away'

# Try to get score from JSON
try:
    with open(JSON_PATH) as f:
        raw_json = json.load(f)
    score = raw_json.get('matchDetails', {}).get('scores', {}).get('ft', {'home': '?', 'away': '?'})
    SCORE_STR   = f"{score.get('home', '?')} – {score.get('away', '?')}"
    MATCH_TITLE = f"{home_name}  {SCORE_STR}  {away_name}"
except Exception:
    MATCH_TITLE = f"{home_name} vs {away_name}"

# Map IDs to names — most shots team = home (heuristic; override manually if needed)
from collections import Counter
team_ids = shots['contestant_id'].unique()
cnt = Counter(shots['contestant_id'])
home_id = cnt.most_common(1)[0][0]
away_id = [t for t in team_ids if t != home_id][0] if len(team_ids) > 1 else home_id

id_to_name = {home_id: home_name, away_id: away_name}
shots['team'] = shots['contestant_id'].map(id_to_name).fillna('Unknown')

print(f'Match: {MATCH_TITLE}')
for tid, name in [(home_id, home_name), (away_id, away_name)]:
    t = shots[shots.contestant_id == tid]
    print(f'  {name}: shots={len(t)}  goals={int(t.is_goal.sum())}  '
          f'xG={t.xg.sum():.2f}  psxG={t.psxg.sum():.2f}')

## 3 · Full shot table

In [ ]:
display_cols = [
    'team', 'player_name', 'time_min', 'outcome',
    'x', 'y', 'distance', 'angle',
    'xg', 'psxg', 'placement_score', 'shot_power',
    'placement_quality', 'finishing_luck',
    'is_header', 'is_volley', 'is_first_time', 'under_pressure',
    'is_fast_break', 'is_from_corner', 'is_open_play',
    'is_outfield_block', 'is_keeper_save',
    'goal_y_norm', 'goal_h_norm', 'goal_zone',
]
display_cols = [c for c in display_cols if c in shots.columns]
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
shots[display_cols].round(3)

## 4 · xG Shot Maps

In [ ]:
BG, PITCH_LINE = '#0d1b2a', '#3a5068'
OUTCOME_COLORS = {
    'Goal':      '#f1c40f',
    'On Target': '#58a6ff',
    'Post':      '#ffffff',
    'Missed':    '#3a5068',
}

def load_badge(name, zoom=0.07):
    if not BADGE_DIR: return None
    for ext in ['png', 'PNG', 'jpg', 'jpeg']:
        p = os.path.join(BADGE_DIR, f'{name}.{ext}')
        if os.path.exists(p):
            try: return OffsetImage(mpimg.imread(p), zoom=zoom)
            except: pass
    return None

def add_badge(ax, name, pos=(0.06, 0.97)):
    img = load_badge(name)
    if img:
        ax.add_artist(AnnotationBbox(img, pos, xycoords='axes fraction', frameon=False))

def plot_xg_map(team_id, subtitle_label, save_suffix=None):
    t = shots[shots.contestant_id == team_id].copy()
    goals, xg, psxg = int(t.is_goal.sum()), t.xg.sum(), t.psxg.sum()

    fig = plt.figure(figsize=(14, 11)); fig.patch.set_facecolor(BG)
    gs  = fig.add_gridspec(3, 1, height_ratios=[1.2, 13, 2.0], hspace=0.04)
    ax_t, ax_p, ax_f = fig.add_subplot(gs[0]), fig.add_subplot(gs[1]), fig.add_subplot(gs[2])
    for ax in [ax_t, ax_f]: ax.set_facecolor(BG); ax.axis('off')

    ax_t.text(0.5, 0.82, MATCH_TITLE, ha='center', va='center', fontsize=20,
              color='white', fontweight='bold', transform=ax_t.transAxes)
    ax_t.text(0.5, 0.20, f'{subtitle_label}  —  xG Shot Map',
              ha='center', va='center', fontsize=11, color='#7a9ab0', transform=ax_t.transAxes)

    ax_f.text(0.02, 0.75,
              f'Goals: {goals}   xG: {xg:.2f}   psxG: {psxg:.2f}   G−xG: {goals - xg:+.2f}',
              ha='left', va='center', fontsize=11, color='white',
              fontweight='bold', transform=ax_f.transAxes)
    for lx, (lbl, lc) in zip([0.45, 0.57, 0.69, 0.80], OUTCOME_COLORS.items()):
        ax_f.scatter([lx], [0.75], s=80, color=lc, transform=ax_f.transAxes, zorder=4, clip_on=False)
        ax_f.text(lx + 0.015, 0.75, lbl, ha='left', va='center', fontsize=9,
                  color='white', transform=ax_f.transAxes)
    ax_f.text(0.5, 0.12, 'Dot size = xG value  ·  Data: Opta  //  Marc Lamberts',
              ha='center', va='center', fontsize=9, color='#7a9ab0',
              fontweight='bold', transform=ax_f.transAxes)

    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                          half=True, pad_top=5, pad_bottom=2, pad_left=4, pad_right=4)
    pitch.draw(ax=ax_p)

    for _, row in t.iterrows():
        oc  = row['outcome']
        col = OUTCOME_COLORS.get(oc, '#3a5068')
        sz  = max(40, row['xg'] * 1800)
        pitch.scatter(row['x'], row['y'], s=sz, color=col,
                      edgecolors='white' if oc == 'Goal' else '#aaaaaa',
                      linewidth=2.0 if oc == 'Goal' else 0.8,
                      ax=ax_p, zorder=5, alpha=0.90)
        pitch.annotate(f"{row['xg']:.2f}", xy=(row['x'], row['y']), ax=ax_p,
                       ha='center', va='bottom', fontsize=7, color='white', alpha=0.75,
                       xytext=(0, int(sz**0.5) + 4), textcoords='offset points', zorder=6)

    add_badge(ax_p, id_to_name.get(team_id, ''))

    if OUT_DIR and save_suffix:
        os.makedirs(OUT_DIR, exist_ok=True)
        plt.savefig(os.path.join(OUT_DIR, f'ShotMap_xG_{save_suffix}.png'),
                    dpi=200, bbox_inches='tight', facecolor=BG)
    plt.show()

plot_xg_map(home_id, home_name, save_suffix=home_name.replace(' ', '_'))
plot_xg_map(away_id, away_name, save_suffix=away_name.replace(' ', '_'))

## 5 · psxG Map (post-shot placement)

In [ ]:
def plot_psxg_map(team_id, subtitle_label, save_suffix=None):
    t = shots[shots.contestant_id == team_id].copy()
    goals, xg, psxg = int(t.is_goal.sum()), t.xg.sum(), t.psxg.sum()

    fig = plt.figure(figsize=(14, 11)); fig.patch.set_facecolor(BG)
    gs  = fig.add_gridspec(3, 1, height_ratios=[1.2, 13, 2.0], hspace=0.04)
    ax_t, ax_p, ax_f = fig.add_subplot(gs[0]), fig.add_subplot(gs[1]), fig.add_subplot(gs[2])
    for ax in [ax_t, ax_f]: ax.set_facecolor(BG); ax.axis('off')

    ax_t.text(0.5, 0.82, MATCH_TITLE, ha='center', va='center', fontsize=20,
              color='white', fontweight='bold', transform=ax_t.transAxes)
    ax_t.text(0.5, 0.20, f'{subtitle_label}  —  psxG Shot Map  (placement quality)',
              ha='center', va='center', fontsize=11, color='#7a9ab0', transform=ax_t.transAxes)

    ax_f.text(0.02, 0.75,
              f'Goals: {goals}   xG: {xg:.2f}   psxG: {psxg:.2f}   psxG−xG: {psxg - xg:+.2f}',
              ha='left', va='center', fontsize=11, color='white',
              fontweight='bold', transform=ax_f.transAxes)
    for lx, (lbl, lc) in zip([0.45, 0.57, 0.69, 0.80], OUTCOME_COLORS.items()):
        ax_f.scatter([lx], [0.75], s=80, color=lc, transform=ax_f.transAxes, zorder=4, clip_on=False)
        ax_f.text(lx + 0.015, 0.75, lbl, ha='left', va='center', fontsize=9,
                  color='white', transform=ax_f.transAxes)
    ax_f.text(0.5, 0.12, 'Dot size = psxG value  ·  Data: Opta  //  Marc Lamberts',
              ha='center', va='center', fontsize=9, color='#7a9ab0',
              fontweight='bold', transform=ax_f.transAxes)

    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                          half=True, pad_top=5, pad_bottom=2, pad_left=4, pad_right=4)
    pitch.draw(ax=ax_p)

    for _, row in t.iterrows():
        oc  = row['outcome']
        col = OUTCOME_COLORS.get(oc, '#3a5068')
        sz  = max(30, row['psxg'] * 1800)
        pitch.scatter(row['x'], row['y'], s=sz, color=col,
                      edgecolors='white' if oc == 'Goal' else '#aaaaaa',
                      linewidth=2.0 if oc == 'Goal' else 0.8,
                      ax=ax_p, zorder=5, alpha=0.90)
        pitch.annotate(f"{row['psxg']:.2f}", xy=(row['x'], row['y']), ax=ax_p,
                       ha='center', va='bottom', fontsize=7, color='white', alpha=0.75,
                       xytext=(0, int(sz**0.5) + 4), textcoords='offset points', zorder=6)

    add_badge(ax_p, id_to_name.get(team_id, ''))

    if OUT_DIR and save_suffix:
        plt.savefig(os.path.join(OUT_DIR, f'ShotMap_psxG_{save_suffix}.png'),
                    dpi=200, bbox_inches='tight', facecolor=BG)
    plt.show()

plot_psxg_map(home_id, home_name, save_suffix=home_name.replace(' ', '_'))
plot_psxg_map(away_id, away_name, save_suffix=away_name.replace(' ', '_'))

## 6 · xG Flow Chart

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5)); fig.patch.set_facecolor(BG); ax.set_facecolor(BG)

HOME_COLOR, AWAY_COLOR = '#E8720C', '#1A78CF'

for team_id, name, color in [(home_id, home_name, HOME_COLOR), (away_id, away_name, AWAY_COLOR)]:
    t = shots[shots.contestant_id == team_id].sort_values('time_min').copy()
    t['cum_xg'] = t['xg'].cumsum()
    ax.step(t['time_min'], t['cum_xg'], where='post', color=color, lw=2, label=f'{name} xG')
    goals_t = t[t.is_goal == 1]
    ax.scatter(goals_t['time_min'], goals_t['cum_xg'], color=color, s=120,
               zorder=5, marker='*', edgecolors='white', linewidth=1)

ax.axvline(45, color='white', lw=0.8, alpha=0.3, ls='--')
ax.set_xlabel('Minute', color='white'); ax.set_ylabel('Cumulative xG', color='white')
ax.tick_params(colors='white'); ax.spines[:].set_color('#3a5068')
ax.set_title('xG Flow', color='white', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, facecolor=BG, labelcolor='white')
ax.text(0.99, 0.02, 'Data: Opta  //  Marc Lamberts', transform=ax.transAxes,
        ha='right', va='bottom', fontsize=8, color='#7a9ab0')

if OUT_DIR:
    plt.savefig(os.path.join(OUT_DIR, 'xGFlow.png'), dpi=200, bbox_inches='tight', facecolor=BG)
plt.show()

## 7 · Export shot CSV

In [ ]:
export_cols = [
    'player_name', 'team', 'contestant_id', 'period_id', 'time_min',
    'x', 'y', 'distance', 'angle', 'open_angle',
    'outcome', 'is_goal', 'is_on_target', 'is_blocked', 'is_post',
    'is_outfield_block', 'is_keeper_save',
    'xg', 'psxg',
    'placement_score', 'shot_power', 'placement_quality', 'finishing_luck',
    'goal_y_norm', 'goal_h_norm', 'goal_zone',
    'is_header', 'is_right_foot', 'is_left_foot', 'weak_foot',
    'is_volley', 'is_deflected', 'is_first_time', 'is_big_chance',
    'is_fast_break', 'is_from_corner', 'is_free_kick', 'is_penalty',
    'is_set_piece', 'is_open_play', 'is_pull_back', 'under_pressure',
]
export_cols = [c for c in export_cols if c in shots.columns]

if OUT_DIR:
    out_path = os.path.join(OUT_DIR, f'{filename_base}_shots_xg.csv')
    shots[export_cols].to_csv(out_path, index=False)
    print(f'Saved → {out_path}')
else:
    print('Set OUT_DIR to save CSV')

shots[export_cols].round(3)